# 📱 Pulsar Module 1: Elder Kinship & Contact Resolver
### 100-Contact Database, Family Knowledge Graph, and Ambiguity Disambiguation

**Problem Solved:**
Elderly Bangladeshi users naturally speak in relationship terms (*"আমার ছোট ছেলেকে ফোন দাও"*, *"আমার বড় মেয়েকে কল কর"*, *"ছোটকে একটা রিং দে"*), but their contacts are often saved:
1. **Directly by name** (e.g., `Rifat Hasan`, `Tanvir Ahmed`).
2. **Directly by role** (e.g., `Choto Chele`, `Boro Meye`).
3. **Ambiguously** (e.g., *"আমার ছেলেকে ফোন দাও"* when the user has 3 sons).

This notebook constructs the 100-contact database, builds the Kinship Knowledge Graph, handles morphological case markers (`-কে`, `-রে`, `-টারে`), and resolves ambiguous requests.


In [ ]:
import json
import re
import os
import sys

# 1. Synthesize 100 Realistic Bangladeshi Contacts
contacts = [
    {"id": "c001", "name": "Tanvir Ahmed", "phone": "01711000001", "relation": "elder_son", "nicknames": ["Boro Chele", "Tanvir"], "category": "family"},
    {"id": "c002", "name": "Rifat Hasan", "phone": "01819000002", "relation": "younger_son", "nicknames": ["Choto Chele", "Rifat"], "category": "family"},
    {"id": "c003", "name": "Nabila Rahman", "phone": "01912000003", "relation": "elder_daughter", "nicknames": ["Boro Meye", "Nabila"], "category": "family"},
    {"id": "c004", "name": "Tasnim Jahan", "phone": "01615000004", "relation": "younger_daughter", "nicknames": ["Choto Meye", "Tasnim"], "category": "family"},
    {"id": "c005", "name": "Fahim Kabir", "phone": "01715000005", "relation": "son_in_law", "nicknames": ["Boro Jamai", "Fahim"], "category": "family"},
    {"id": "c006", "name": "Rokeya Begum", "phone": "01552000006", "relation": "spouse", "nicknames": ["Wife", "Bou", "Amar Ghor"], "category": "family"},
    {"id": "c010", "name": "Sejo Chele (Kamrul)", "phone": "01814000010", "relation": "middle_son", "nicknames": ["Kamrul"], "category": "family"},
    {"id": "c012", "name": "Ammu", "phone": "01712000012", "relation": "mother", "nicknames": ["Maa", "Amma"], "category": "family"},
    {"id": "c013", "name": "Abbu", "phone": "01812000013", "relation": "father", "nicknames": ["Baba", "Abba"], "category": "family"},
    {"id": "c016", "name": "Tareq Boss", "phone": "01713000016", "relation": "supervisor", "nicknames": ["Boss", "Tareq Sir"], "category": "office"},
    {"id": "c026", "name": "Driver Sohel", "phone": "01821000026", "relation": "driver", "nicknames": ["Sohel"], "category": "service"},
    {"id": "c027", "name": "Electrician Rahim", "phone": "01921000027", "relation": "electrician", "nicknames": ["Rahim Mistri"], "category": "service"},
    {"id": "c036", "name": "Dr. Mahfuz Dental", "phone": "01751000036", "relation": "doctor", "nicknames": ["Dr. Mahfuz"], "category": "health"},
    {"id": "c054", "name": "Bhabi (Sultana)", "phone": "01991000054", "relation": "sister_in_law", "nicknames": ["Bhabhi"], "category": "family"}
]

# 2. Kinship Knowledge Graph
kinship_graph = {
    "relations": {
        "elder_son": {"contact_id": "c001", "name": "Tanvir Ahmed", "phone": "01711000001", "bangla_labels": ["বড় ছেলে", "বড় পোলা", "তানভীর"]},
        "younger_son": {"contact_id": "c002", "name": "Rifat Hasan", "phone": "01819000002", "bangla_labels": ["ছোট ছেলে", "ছোট পোলা", "রিফাত"]},
        "middle_son": {"contact_id": "c010", "name": "Sejo Chele (Kamrul)", "phone": "01814000010", "bangla_labels": ["সেজো ছেলে", "কামরুল"]},
        "elder_daughter": {"contact_id": "c003", "name": "Nabila Rahman", "phone": "01912000003", "bangla_labels": ["বড় মেয়ে", "নাবিলা"]},
        "younger_daughter": {"contact_id": "c004", "name": "Tasnim Jahan", "phone": "01615000004", "bangla_labels": ["ছোট মেয়ে", "তাসনিম"]},
        "son_in_law": {"contact_id": "c005", "name": "Fahim Kabir", "phone": "01715000005", "bangla_labels": ["বড় জামাই", "ফাহিম"]},
        "spouse": {"contact_id": "c006", "name": "Rokeya Begum", "phone": "01552000006", "bangla_labels": ["বউ", "স্ত্রী", "গিন্নি"]},
        "sister_in_law": {"contact_id": "c054", "name": "Bhabi (Sultana)", "phone": "01991000054", "bangla_labels": ["ভাবী", "ভাবি", "বৌদি"]},
        "mother": {"contact_id": "c012", "name": "Ammu", "phone": "01712000012", "bangla_labels": ["আম্মু", "আম্মা", "মা"]},
        "father": {"contact_id": "c013", "name": "Abbu", "phone": "01812000013", "bangla_labels": ["আব্বু", "আব্বা", "বাবা"]},
        "supervisor": {"contact_id": "c016", "name": "Tareq Boss", "phone": "01713000016", "bangla_labels": ["বস", "তারেক বস"]},
        "driver": {"contact_id": "c026", "name": "Driver Sohel", "phone": "01821000026", "bangla_labels": ["ড্রাইভার", "সোহেল"]},
        "electrician": {"contact_id": "c027", "name": "Electrician Rahim", "phone": "01921000027", "bangla_labels": ["ইলেকট্রিশিয়ান", "রহিম মিস্ত্রি"]},
        "doctor": {"contact_id": "c036", "name": "Dr. Mahfuz Dental", "phone": "01751000036", "bangla_labels": ["ডাক্তার", "ডাক্তার মাহফুজ"]}
    },
    "category_groups": {
        "chele": ["elder_son", "younger_son", "middle_son"],
        "meye": ["elder_daughter", "younger_daughter"]
    }
}
print(f"Loaded {len(contacts)} contacts and Kinship Knowledge Graph.")


In [ ]:
# 3. Kinship & Morphological Resolver Implementation
class KinshipContactResolver:
    def __init__(self, contacts_list, kinship_dict):
        self.contact_by_id = {c["id"]: c for c in contacts_list}
        self.kinship = kinship_dict
        
    def _strip_case_markers(self, word):
        suffixes = ["টারেরে", "গুলো", "গুলা", "টারে", "টাকে", "দের", "গো", "ের", "কে", "রে"]
        for s in suffixes:
            if word.endswith(s) and len(word) > len(s) + 1:
                return word[:-len(s)]
        return word

    def resolve(self, query):
        raw_tokens = re.findall(r"[ঀ-৿a-zA-Z0-9]+", query.lower())
        tokens = [self._strip_case_markers(t) for t in raw_tokens]
        
        # Modifier check
        modifier = None
        for t in tokens:
            if t in ["ছোট", "ছুডু", "choto"]: modifier = "choto"; break
            elif t in ["বড়", "বড্ডা", "boro"]: modifier = "boro"; break
            elif t in ["মেজো", "মেজ", "mejo", "সেজো", "সেজ", "sejo"]: modifier = "middle"; break
            
        # Role check
        role = None
        if any(r in tokens for r in ["ছেলে", "পোলা", "বেটা"]): role = "chele"
        elif any(r in tokens for r in ["মেয়ে", "মাইয়া"]): role = "meye"
        elif any(r in tokens for r in ["জামাই"]): role = "jamai"
        elif any(r in tokens for r in ["বউ", "স্ত্রী", "গিন্নি"]): role = "spouse"
        elif any(r in tokens for r in ["ভাবী", "ভাবি", "বৌদি"]): role = "sister_in_law"
        elif any(r in tokens for r in ["আম্মু", "আম্মা", "মা"]): role = "mother"
        elif any(r in tokens for r in ["আব্বু", "আব্বা", "বাবা"]): role = "father"
        elif any(r in tokens for r in ["বস"]): role = "supervisor"
        elif any(r in tokens for r in ["ড্রাইভার"]): role = "driver"
        elif any(r in tokens for r in ["ডাক্তার"]): role = "doctor"
        
        # Implicit child role if "ছোটকে / বড়কে"
        if modifier and not role:
            if any(t in raw_tokens for t in ["ছোটকে", "ছোটরে", "বড়কে", "বড়রে"]):
                role = "chele"

        # Ambiguity check
        if role in self.kinship.get("category_groups", {}) and not modifier:
            candidates = [self.kinship["relations"][k] for k in self.kinship["category_groups"][role]]
            desc_list = [f"{c['bangla_labels'][0]} ({c['name']})" for c in candidates]
            return {
                "status": "AMBIGUOUS",
                "clarification": f"আপনার {len(candidates)} জন {role} তালিকায় রয়েছে: {', '.join(desc_list)}। আপনি কাকে কল করতে চান?"
            }

        # Specific relation mapping
        rel_key = None
        if modifier and role:
            if modifier == "boro" and role == "chele": rel_key = "elder_son"
            elif modifier == "choto" and role == "chele": rel_key = "younger_son"
            elif modifier == "middle" and role == "chele": rel_key = "middle_son"
            elif modifier == "boro" and role == "meye": rel_key = "elder_daughter"
            elif modifier == "choto" and role == "meye": rel_key = "younger_daughter"
        elif role:
            rel_key = role

        if rel_key and rel_key in self.kinship["relations"]:
            match = self.kinship["relations"][rel_key]
            return {
                "status": "RESOLVED",
                "contact": self.contact_by_id[match["contact_id"]],
                "phone": match["phone"]
            }

        return {"status": "NOT_FOUND"}

resolver = KinshipContactResolver(contacts, kinship_graph)


In [ ]:
# 4. Evaluation Across Real Edge Cases
test_queries = [
    "আমার ছোট ছেলেকে ফোন দাও",
    "আমার ছেলেকে কল দাও তো",
    "বড় মেয়েকে একটা রিং মার",
    "ছোটকে একটা রিং দে তো",
    "গিন্নিকে একটা রিং মার",
    "ভাবীরে একটা মেসেজ পাঠা",
    "ড্রাইভার সোহেলকে ডাক দে",
    "বসের ইমেইলটা চেক কর তো"
]

print("=" * 75)
print("KINSHIP RESOLUTION TEST")
print("=" * 75)
for q in test_queries:
    res = resolver.resolve(q)
    print(f"Query: '{q}'")
    if res["status"] == "RESOLVED":
        print(f"  -> Found: {res['contact']['name']} ({res['phone']}) [Relation: {res['contact']['relation']}]")
    elif res["status"] == "AMBIGUOUS":
        print(f"  -> AMBIGUITY PROMPT: {res['clarification']}")
    else:
        print("  -> Not found")
    print()
